In [ ]:
!pip install gcloud
!gcloud auth application-default login
import pandas as pd
import numpy as np
import os
import pandas_gbq
from google.cloud import bigquery
import glob
import openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.4/454.4 kB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gcloud: filename=gcloud-0.18.3-py3-none-any.whl size=602927 sha256=6283c9fbbc6aca6de998aff4f671df799ea2a213cf1dd4036bafb88510fa3a23
  Stored in directory: /root/.cache/pip/wheels/2a/62/75/3d74209bfebb8805823ae74afa28653aa1ea76d8b5a9d741ff
Successfully built gcloud
Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=RBkXHT9rWOQ7yyEGxflQGonIzSuqLw&prompt=consent&token_usage=remote&access_type=offline&code_chal


# 2019

In [ ]:
dfseg19 = pd.read_excel('/content/Base_MUNIC_2019_20210817.xlsx', sheet_name='Segurança pública', usecols=['CodMun', 'COD UF','NOME MUNIC','MSEG01','MSEG03', 'MSEG04', 'MSEG05', 'MSEG06'])
dfseg19['ano']= 2019
uf = pd.read_excel('/content/Base_MUNIC_2019_20210817.xlsx', sheet_name = 'Variáveis externas', usecols=[2,3]) # Pegando nome e codigo das UF
dfseg19 = dfseg19.rename(columns={'COD UF':'cod_uf',
                                'CodMun':'cod_municipio',
                                'NOME MUNIC':'nome_municipio',
                                'MSEG01':'secretaria',
                                'MSEG03':'genero',
                                'MSEG04':'idade',
                                'MSEG05':'cor_raca',
                                'MSEG06':'grau_instrucao'
                             })


In [ ]:
uf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   COD UF  5570 non-null   int64 
 1   UF      5570 non-null   object
dtypes: int64(1), object(1)
memory usage: 87.2+ KB


In [ ]:
x = uf.pivot_table(columns=('UF','COD UF'), aggfunc='size')

In [ ]:
uf = pd.DataFrame(x).reset_index()[['UF','COD UF']]

In [ ]:
dfseg19 = dfseg19.merge(uf, right_on='COD UF',left_on='cod_uf')
dfseg19

,cod_municipio,cod_uf,nome_municipio,secretaria,genero,idade,cor_raca,grau_instrucao,ano,UF,COD UF
0,1100015,11,Alta Floresta D'Oeste,Não possui estrutura,-,-,-,-,2019,Rondônia,11
1,1100023,11,Ariquemes,Secretaria municipal em conjunto com outras po...,Masculino,39,Branca,Especialização,2019,Rondônia,11
2,1100031,11,Cabixi,Não possui estrutura,-,-,-,-,2019,Rondônia,11
3,1100049,11,Cacoal,Não possui estrutura,-,-,-,-,2019,Rondônia,11
4,1100056,11,Cerejeiras,Não possui estrutura,-,-,-,-,2019,Rondônia,11
...,...,...,...,...,...,...,...,...,...,...,...
5565,5222005,52,Vianópolis,Não possui estrutura,-,-,-,-,2019,Goiás,52
5566,5222054,52,Vicentinópolis,Não possui estrutura,-,-,-,-,2019,Goiás,52
5567,5222203,52,Vila Boa,Não possui estrutura,-,-,-,-,2019,Goiás,52
5568,5222302,52,Vila Propício,Não possui estrutura,-,-,-,-,2019,Goiás,52


In [ ]:
dfseg19.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   cod_municipio   5570 non-null   int64 
 1   cod_uf          5570 non-null   int64 
 2   nome_municipio  5570 non-null   object
 3   secretaria      5570 non-null   object
 4   genero          5570 non-null   object
 5   idade           5570 non-null   object
 6   cor_raca        5570 non-null   object
 7   grau_instrucao  5570 non-null   object
 8   ano             5570 non-null   int64 
 9   UF              5570 non-null   object
 10  COD UF          5570 non-null   int64 
dtypes: int64(4), object(7)
memory usage: 478.8+ KB


In [ ]:
dfseg19 = dfseg19.drop(['COD UF'], axis=1) #eliminando coluna repetida

In [ ]:
dfseg19.head(2)

,cod_municipio,cod_uf,nome_municipio,secretaria,genero,idade,cor_raca,grau_instrucao,ano,UF
0,1100015,11,Alta Floresta D'Oeste,Não possui estrutura,-,-,-,-,2019,Rondônia
1,1100023,11,Ariquemes,Secretaria municipal em conjunto com outras po...,Masculino,39,Branca,Especialização,2019,Rondônia


In [ ]:
dfseg19 = dfseg19.rename(columns={'codigo_uf':'cod_uf', 'UF':'sigla_uf', 'cod_municipio':'id_municipio'})
dfseg19.head(2)

,id_municipio,cod_uf,nome_municipio,secretaria,genero,idade,cor_raca,grau_instrucao,ano,sigla_uf
0,1100015,11,Alta Floresta D'Oeste,Não possui estrutura,-,-,-,-,2019,Rondônia
1,1100023,11,Ariquemes,Secretaria municipal em conjunto com outras po...,Masculino,39,Branca,Especialização,2019,Rondônia


In [ ]:
dfseg19.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_municipio    5570 non-null   int64 
 1   cod_uf          5570 non-null   int64 
 2   nome_municipio  5570 non-null   object
 3   secretaria      5570 non-null   object
 4   genero          5570 non-null   object
 5   idade           5570 non-null   object
 6   cor_raca        5570 non-null   object
 7   grau_instrucao  5570 non-null   object
 8   ano             5570 non-null   int64 
 9   sigla_uf        5570 non-null   object
dtypes: int64(3), object(7)
memory usage: 435.3+ KB


In [ ]:
dfseg19['cor_raca'] = np.where(dfseg19['cor_raca'] == 'Pardo', 'Parda', dfseg19['cor_raca'])

# Handle missing/refused data for gender column
dfseg19['genero'] = np.where(dfseg19['genero'] == 'Recusa', 'Sem dados', dfseg19['genero'])
dfseg19['genero'] = np.where(dfseg19['genero'] == 'Não informou', 'Sem dados', dfseg19['genero'])
dfseg19['genero'] = np.where(dfseg19['genero'] == '-', 'Sem dados', dfseg19['genero'])
dfseg19['genero'] = np.where(dfseg19['genero'] == '(**) Sem gestor', 'Sem dados', dfseg19['genero'])
dfseg19['genero'] = np.where(dfseg19['genero'] == 'Não soube informar', 'Sem dados', dfseg19['genero'])
dfseg19['genero'] = np.where(dfseg19['genero'] == '(*) Não soube informar', 'Sem dados', dfseg19['genero'])
dfseg19['genero'] = np.where(dfseg19['genero'] == 'Sem titular', 'Sem dados', dfseg19['genero'])

# Handle missing/refused data for race/color column
dfseg19['cor_raca'] = np.where(dfseg19['cor_raca'] == 'Recusa', 'Sem dados', dfseg19['cor_raca'])
dfseg19['cor_raca'] = np.where(dfseg19['cor_raca'] == 'Não informou', 'Sem dados', dfseg19['cor_raca'])
dfseg19['cor_raca'] = np.where(dfseg19['cor_raca'] == '-', 'Sem dados', dfseg19['cor_raca'])
dfseg19['cor_raca'] = np.where(dfseg19['cor_raca'] == '(**) Sem gestor', 'Sem dados', dfseg19['cor_raca'])
dfseg19['cor_raca'] = np.where(dfseg19['cor_raca'] == 'Não soube informar', 'Sem dados', dfseg19['cor_raca'])
dfseg19['cor_raca'] = np.where(dfseg19['cor_raca'] == '(*) Não soube informar', 'Sem dados', dfseg19['cor_raca'])
dfseg19['cor_raca'] = np.where(dfseg19['cor_raca'] == 'Sem titular', 'Sem dados', dfseg19['cor_raca'])

# Handle missing/refused data for education level column
dfseg19['grau_instrucao'] = np.where(dfseg19['grau_instrucao'] == 'Recusa', 'Sem dados', dfseg19['grau_instrucao'])
dfseg19['grau_instrucao'] = np.where(dfseg19['grau_instrucao'] == 'Não informou', 'Sem dados', dfseg19['grau_instrucao'])
dfseg19['grau_instrucao'] = np.where(dfseg19['grau_instrucao'] == '-', 'Sem dados', dfseg19['grau_instrucao'])
dfseg19['grau_instrucao'] = np.where(dfseg19['grau_instrucao'] == '(**) Sem gestor', 'Sem dados', dfseg19['grau_instrucao'])
dfseg19['grau_instrucao'] = np.where(dfseg19['grau_instrucao'] == 'Não soube informar', 'Sem dados', dfseg19['grau_instrucao'])
dfseg19['grau_instrucao'] = np.where(dfseg19['grau_instrucao'] == '(*) Não soube informar', 'Sem dados', dfseg19['grau_instrucao'])
dfseg19['grau_instrucao'] = np.where(dfseg19['grau_instrucao'] == 'Sem titular', 'Sem dados', dfseg19['grau_instrucao'])

# Handle missing/refused data for age column (convert to NaN)
dfseg19['idade'] = np.where(dfseg19['idade'] == 'Recusa', np.nan, dfseg19['idade'])
dfseg19['idade'] = np.where(dfseg19['idade'] == 'Não informou', np.nan, dfseg19['idade'])
dfseg19['idade'] = np.where(dfseg19['idade'] == '-', np.nan, dfseg19['idade'])
dfseg19['idade'] = np.where(dfseg19['idade'] == '(**) Sem gestor', np.nan, dfseg19['idade'])
dfseg19['idade'] = np.where(dfseg19['idade'] == 'Não soube informar', np.nan, dfseg19['idade'])
dfseg19['idade'] = np.where(dfseg19['idade'] == '(*) Não soube informar', np.nan, dfseg19['idade'])
dfseg19['idade'] = np.where(dfseg19['idade'] == 'Sem titular', np.nan, dfseg19['idade'])

# Convert age column to numeric type
dfseg19['idade'] = pd.to_numeric(dfseg19['idade'])

# Define age group bins and labels
limites = [18, 30, 50, 65, 100]
categorias = ['Entre 18-29', 'Entre 30-49', 'Entre 50-64', 'Acima de 65']

# Create age group column based on the bins
dfseg19['faixa_etaria'] = pd.cut(dfseg19['idade'], bins=limites, labels=categorias, right=False)

# Reorder columns in the dataframe
dfseg19 = dfseg19[['ano', 'sigla_uf', 'nome_municipio', 'id_municipio', 'genero', 'cor_raca', 'grau_instrucao', 'faixa_etaria', 'secretaria']]

# Create dictionary to standardize education levels
dict_esco = {
    'Ensino fundamental incompleto': 'Até Ensino Fundamental',
    'Ensino fundamental completo': 'Até Ensino Fundamental',
    'Ensino fundamental (1º Grau) completo': 'Até Ensino Fundamental',
    'Ensino fundamental (1º Grau) incompleto': 'Até Ensino Fundamental',
    'Ensino médio (2º Grau) incompleto': 'Até Ensino Médio',
    'Ensino médio (2º Grau) completo': 'Até Ensino Médio',
    'Ensino médio completo': 'Até Ensino Médio',
    'Ensino superior incompleto': 'Até Ensino Superior Completo',
    'Ensino superior completo': 'Até Ensino Superior Completo',
    'Especialização': 'Até Pós Graduação ou Mestrado',
    'Mestrado': 'Até Pós Graduação ou Mestrado',
    'Doutorado': 'Até Doutorado'
}

# Apply education level standardization
dfseg19 = dfseg19.replace({'grau_instrucao': dict_esco})

# Display unique values in education column for verification
dfseg19['grau_instrucao'].unique()

array(['Sem dados', 'Até Pós Graduação ou Mestrado',
       'Até Ensino Superior Completo', 'Até Ensino Médio',
       'Até Ensino Fundamental', 'Ensino médio incompleto',
       'Até Doutorado'], dtype=object)

In [ ]:
dfseg19.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   ano             5570 non-null   int64   
 1   sigla_uf        5570 non-null   object  
 2   nome_municipio  5570 non-null   object  
 3   id_municipio    5570 non-null   int64   
 4   genero          5570 non-null   object  
 5   cor_raca        5570 non-null   object  
 6   grau_instrucao  5570 non-null   object  
 7   faixa_etaria    1298 non-null   category
 8   secretaria      5570 non-null   object  
dtypes: category(1), int64(2), object(6)
memory usage: 353.9+ KB



# 2023

In [ ]:
dfseg23 = pd.read_excel('/content/Base_MUNIC_2023.xlsx', sheet_name='Segurança Pública', usecols=['CodMun', 'Cod UF', 'UF', 'Mun','MSEG01','MSEG03', 'MSEG04', 'MSEG05', 'MSEG06'])
dfseg23['ano']= 2023
uf = pd.read_excel('/content/Base_MUNIC_2019_20210817.xlsx', sheet_name = 'Variáveis externas', usecols=[2,3]) # Pegando nome e codigo das UF
dfseg23 = dfseg23.rename(columns={'Cod UF':'cod_uf',
                                'CodMun':'cod_municipio',
                                'Mun':'nome_municipio',
                                'MSEG01':'secretaria',
                                'MSEG03':'genero',
                                'MSEG04':'idade',
                                'MSEG05':'cor_raca',
                                'MSEG06':'grau_instrucao'
                             })


In [ ]:
dfseg23.head(2)

,cod_municipio,UF,cod_uf,nome_municipio,secretaria,genero,idade,cor_raca,grau_instrucao,ano
0,1100015,RO,11,Alta Floresta DOeste,Não possui estrutura,-,-,-,-,2023
1,1100023,RO,11,Ariquemes,Secretaria municipal em conjunto com outras po...,Masculino,34,Branca,Especialização,2023


In [ ]:
uf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   COD UF  5570 non-null   int64 
 1   UF      5570 non-null   object
dtypes: int64(1), object(1)
memory usage: 87.2+ KB


In [ ]:
x = uf.pivot_table(columns=('UF','COD UF'), aggfunc='size')

In [ ]:
uf = pd.DataFrame(x).reset_index()[['UF','COD UF']]

In [ ]:
dfseg23 = dfseg23.merge(uf, right_on='COD UF',left_on='cod_uf')
dfseg23

,cod_municipio,UF_x,cod_uf,nome_municipio,secretaria,genero,idade,cor_raca,grau_instrucao,ano,UF_y,COD UF
0,1100015,RO,11,Alta Floresta DOeste,Não possui estrutura,-,-,-,-,2023,Rondônia,11
1,1100023,RO,11,Ariquemes,Secretaria municipal em conjunto com outras po...,Masculino,34,Branca,Especialização,2023,Rondônia,11
2,1100031,RO,11,Cabixi,Não possui estrutura,-,-,-,-,2023,Rondônia,11
3,1100049,RO,11,Cacoal,Não possui estrutura,-,-,-,-,2023,Rondônia,11
4,1100056,RO,11,Cerejeiras,Não possui estrutura,-,-,-,-,2023,Rondônia,11
...,...,...,...,...,...,...,...,...,...,...,...,...
5565,5222005,GO,52,Vianópolis,Setor subordinado a outra secretaria,Masculino,40,Branca,Ensino superior completo,2023,Goiás,52
5566,5222054,GO,52,Vicentinópolis,Não possui estrutura,-,-,-,-,2023,Goiás,52
5567,5222203,GO,52,Vila Boa,Não possui estrutura,-,-,-,-,2023,Goiás,52
5568,5222302,GO,52,Vila Propício,Não possui estrutura,-,-,-,-,2023,Goiás,52


In [ ]:
dfseg23.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   cod_municipio   5570 non-null   int64 
 1   UF_x            5570 non-null   object
 2   cod_uf          5570 non-null   int64 
 3   nome_municipio  5570 non-null   object
 4   secretaria      5570 non-null   object
 5   genero          5570 non-null   object
 6   idade           5570 non-null   object
 7   cor_raca        5570 non-null   object
 8   grau_instrucao  5570 non-null   object
 9   ano             5570 non-null   int64 
 10  UF_y            5570 non-null   object
 11  COD UF          5570 non-null   int64 
dtypes: int64(4), object(8)
memory usage: 522.3+ KB


In [ ]:
dfseg23 = dfseg23.drop(['COD UF'], axis=1) #eliminando coluna repetida

In [ ]:
dfseg23.head(2)

,cod_municipio,UF_x,cod_uf,nome_municipio,secretaria,genero,idade,cor_raca,grau_instrucao,ano,UF_y
0,1100015,RO,11,Alta Floresta DOeste,Não possui estrutura,-,-,-,-,2023,Rondônia
1,1100023,RO,11,Ariquemes,Secretaria municipal em conjunto com outras po...,Masculino,34,Branca,Especialização,2023,Rondônia


In [ ]:
dfseg23 = dfseg23.rename(columns={'codigo_uf':'cod_uf', 'UF_x':'sigla_uf', 'cod_municipio':'id_municipio'})
dfseg23.head(2)

,id_municipio,sigla_uf,cod_uf,nome_municipio,secretaria,genero,idade,cor_raca,grau_instrucao,ano,UF_y
0,1100015,RO,11,Alta Floresta DOeste,Não possui estrutura,-,-,-,-,2023,Rondônia
1,1100023,RO,11,Ariquemes,Secretaria municipal em conjunto com outras po...,Masculino,34,Branca,Especialização,2023,Rondônia


In [ ]:
dfseg19.head(2)

,ano,sigla_uf,nome_municipio,id_municipio,genero,cor_raca,grau_instrucao,faixa_etaria,secretaria
0,2019,Rondônia,Alta Floresta D'Oeste,1100015,Sem dados,Sem dados,Sem dados,NaN,Não possui estrutura
1,2019,Rondônia,Ariquemes,1100023,Masculino,Branca,Até Pós Graduação ou Mestrado,Entre 30-49,Secretaria municipal em conjunto com outras po...


In [ ]:
# Mapeia id_municipio -> sigla correta a partir do segundo df
mapa_uf = dfseg23.set_index('id_municipio')['sigla_uf']

# Substitui a coluna sigla_uf do primeiro df
dfseg19['sigla_uf'] = dfseg19['id_municipio'].map(mapa_uf)

In [ ]:
dfseg23.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_municipio    5570 non-null   int64 
 1   sigla_uf        5570 non-null   object
 2   cod_uf          5570 non-null   int64 
 3   nome_municipio  5570 non-null   object
 4   secretaria      5570 non-null   object
 5   genero          5570 non-null   object
 6   idade           5570 non-null   object
 7   cor_raca        5570 non-null   object
 8   grau_instrucao  5570 non-null   object
 9   ano             5570 non-null   int64 
 10  UF_y            5570 non-null   object
dtypes: int64(3), object(8)
memory usage: 478.8+ KB


In [ ]:
dfseg23['cor_raca'] = np.where(dfseg23['cor_raca'] == 'Pardo', 'Parda', dfseg23['cor_raca'])

# Handle missing/refused data for gender column
dfseg23['genero'] = np.where(dfseg23['genero'] == 'Recusa', 'Sem dados', dfseg23['genero'])
dfseg23['genero'] = np.where(dfseg23['genero'] == 'Não informou', 'Sem dados', dfseg23['genero'])
dfseg23['genero'] = np.where(dfseg23['genero'] == '-', 'Sem dados', dfseg23['genero'])
dfseg23['genero'] = np.where(dfseg23['genero'] == '(**) Sem gestor', 'Sem dados', dfseg23['genero'])
dfseg23['genero'] = np.where(dfseg23['genero'] == 'Não soube informar', 'Sem dados', dfseg23['genero'])
dfseg23['genero'] = np.where(dfseg23['genero'] == '(*) Não soube informar', 'Sem dados', dfseg23['genero'])
dfseg23['genero'] = np.where(dfseg23['genero'] == 'Sem titular', 'Sem dados', dfseg23['genero'])

# Handle missing/refused data for race/color column
dfseg23['cor_raca'] = np.where(dfseg23['cor_raca'] == 'Recusa', 'Sem dados', dfseg23['cor_raca'])
dfseg23['cor_raca'] = np.where(dfseg23['cor_raca'] == 'Não informou', 'Sem dados', dfseg23['cor_raca'])
dfseg23['cor_raca'] = np.where(dfseg23['cor_raca'] == '-', 'Sem dados', dfseg23['cor_raca'])
dfseg23['cor_raca'] = np.where(dfseg23['cor_raca'] == '(**) Sem gestor', 'Sem dados', dfseg23['cor_raca'])
dfseg23['cor_raca'] = np.where(dfseg23['cor_raca'] == 'Não soube informar', 'Sem dados', dfseg23['cor_raca'])
dfseg23['cor_raca'] = np.where(dfseg23['cor_raca'] == '(*) Não soube informar', 'Sem dados', dfseg23['cor_raca'])
dfseg23['cor_raca'] = np.where(dfseg23['cor_raca'] == 'Sem titular', 'Sem dados', dfseg23['cor_raca'])

# Handle missing/refused data for education level column
dfseg23['grau_instrucao'] = np.where(dfseg23['grau_instrucao'] == 'Recusa', 'Sem dados', dfseg23['grau_instrucao'])
dfseg23['grau_instrucao'] = np.where(dfseg23['grau_instrucao'] == 'Não informou', 'Sem dados', dfseg23['grau_instrucao'])
dfseg23['grau_instrucao'] = np.where(dfseg23['grau_instrucao'] == '-', 'Sem dados', dfseg23['grau_instrucao'])
dfseg23['grau_instrucao'] = np.where(dfseg23['grau_instrucao'] == '(**) Sem gestor', 'Sem dados', dfseg23['grau_instrucao'])
dfseg23['grau_instrucao'] = np.where(dfseg23['grau_instrucao'] == 'Não soube informar', 'Sem dados', dfseg23['grau_instrucao'])
dfseg23['grau_instrucao'] = np.where(dfseg23['grau_instrucao'] == '(*) Não soube informar', 'Sem dados', dfseg23['grau_instrucao'])
dfseg23['grau_instrucao'] = np.where(dfseg23['grau_instrucao'] == 'Sem titular', 'Sem dados', dfseg23['grau_instrucao'])

# Handle missing/refused data for age column (convert to NaN)
dfseg23['idade'] = np.where(dfseg23['idade'] == 'Recusa', np.nan, dfseg23['idade'])
dfseg23['idade'] = np.where(dfseg23['idade'] == 'Não informou', np.nan, dfseg23['idade'])
dfseg23['idade'] = np.where(dfseg23['idade'] == '-', np.nan, dfseg23['idade'])
dfseg23['idade'] = np.where(dfseg23['idade'] == '(**) Sem gestor', np.nan, dfseg23['idade'])
dfseg23['idade'] = np.where(dfseg23['idade'] == 'Não soube informar', np.nan, dfseg23['idade'])
dfseg23['idade'] = np.where(dfseg23['idade'] == '(*) Não soube informar', np.nan, dfseg23['idade'])
dfseg23['idade'] = np.where(dfseg23['idade'] == 'Sem titular', np.nan, dfseg23['idade'])

# Convert age column to numeric type
dfseg23['idade'] = pd.to_numeric(dfseg23['idade'])

# Define age group bins and labels
limites = [18, 30, 50, 65, 100]
categorias = ['Entre 18-29', 'Entre 30-49', 'Entre 50-64', 'Acima de 65']

# Create age group column based on the bins
dfseg23['faixa_etaria'] = pd.cut(dfseg23['idade'], bins=limites, labels=categorias, right=False)

# Reorder columns in the dataframe
dfseg23 = dfseg23[['ano', 'sigla_uf', 'nome_municipio', 'id_municipio', 'genero', 'cor_raca', 'grau_instrucao', 'faixa_etaria', 'secretaria']]

# Create dictionary to standardize education levels
dict_esco = {
    'Ensino fundamental incompleto': 'Até Ensino Fundamental',
    'Ensino fundamental completo': 'Até Ensino Fundamental',
    'Ensino fundamental (1º Grau) completo': 'Até Ensino Fundamental',
    'Ensino fundamental (1º Grau) incompleto': 'Até Ensino Fundamental',
    'Ensino médio (2º Grau) incompleto': 'Até Ensino Médio',
    'Ensino médio (2º Grau) completo': 'Até Ensino Médio',
    'Ensino superior incompleto': 'Até Ensino Superior Completo',
    'Ensino superior completo': 'Até Ensino Superior Completo',
    'Especialização': 'Até Pós Graduação ou Mestrado',
    'Mestrado': 'Até Pós Graduação ou Mestrado',
    'Doutorado': 'Até Doutorado'
}

# Apply education level standardization
dfseg23 = dfseg23.replace({'grau_instrucao': dict_esco})

# Display unique values in education column for verification
dfseg23['grau_instrucao'].unique()

array(['Sem dados', 'Até Pós Graduação ou Mestrado', 'Até Ensino Médio',
       'Até Ensino Superior Completo', 'Até Ensino Fundamental',
       'Até Doutorado'], dtype=object)

In [ ]:
dfseg23.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   ano             5570 non-null   int64   
 1   sigla_uf        5570 non-null   object  
 2   nome_municipio  5570 non-null   object  
 3   id_municipio    5570 non-null   int64   
 4   genero          5570 non-null   object  
 5   cor_raca        5570 non-null   object  
 6   grau_instrucao  5570 non-null   object  
 7   faixa_etaria    1704 non-null   category
 8   secretaria      5570 non-null   object  
dtypes: category(1), int64(2), object(6)
memory usage: 353.9+ KB


In [ ]:
dfseg23.head(2)

,ano,sigla_uf,nome_municipio,id_municipio,genero,cor_raca,grau_instrucao,faixa_etaria,secretaria
0,2023,RO,Alta Floresta DOeste,1100015,Sem dados,Sem dados,Sem dados,NaN,Não possui estrutura
1,2023,RO,Ariquemes,1100023,Masculino,Branca,Até Pós Graduação ou Mestrado,Entre 30-49,Secretaria municipal em conjunto com outras po...


# Subindo para o GBQ

In [ ]:
dfseg19.head(2)

,ano,sigla_uf,nome_municipio,id_municipio,genero,cor_raca,grau_instrucao,faixa_etaria,secretaria
0,2019,RO,Alta Floresta D'Oeste,1100015,Sem dados,Sem dados,Sem dados,NaN,Não possui estrutura
1,2019,RO,Ariquemes,1100023,Masculino,Branca,Até Pós Graduação ou Mestrado,Entre 30-49,Secretaria municipal em conjunto com outras po...


In [ ]:
dfseg23.head(2)

,ano,sigla_uf,nome_municipio,id_municipio,genero,cor_raca,grau_instrucao,faixa_etaria,secretaria
0,2023,RO,Alta Floresta DOeste,1100015,Sem dados,Sem dados,Sem dados,NaN,Não possui estrutura
1,2023,RO,Ariquemes,1100023,Masculino,Branca,Até Pós Graduação ou Mestrado,Entre 30-49,Secretaria municipal em conjunto com outras po...


In [ ]:
df_final = pd.concat([dfseg19,dfseg23])
df_final.head(2)

,ano,sigla_uf,nome_municipio,id_municipio,genero,cor_raca,grau_instrucao,faixa_etaria,secretaria
0,2019,RO,Alta Floresta D'Oeste,1100015,Sem dados,Sem dados,Sem dados,NaN,Não possui estrutura
1,2019,RO,Ariquemes,1100023,Masculino,Branca,Até Pós Graduação ou Mestrado,Entre 30-49,Secretaria municipal em conjunto com outras po...


In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11140 entries, 0 to 5569
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   ano             11140 non-null  int64   
 1   sigla_uf        11140 non-null  object  
 2   nome_municipio  11140 non-null  object  
 3   id_municipio    11140 non-null  int64   
 4   genero          11140 non-null  object  
 5   cor_raca        11140 non-null  object  
 6   grau_instrucao  11140 non-null  object  
 7   faixa_etaria    3002 non-null   category
 8   secretaria      11140 non-null  object  
dtypes: category(1), int64(2), object(6)
memory usage: 794.4+ KB


In [ ]:
df_final['ano'].unique()

array([2019, 2023])

In [ ]:
df_final = df_final.rename(columns={'secretaria':'caracterizacao_orgao_gestor'})

In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11140 entries, 0 to 5569
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   ano                          11140 non-null  int64   
 1   sigla_uf                     11140 non-null  object  
 2   nome_municipio               11140 non-null  object  
 3   id_municipio                 11140 non-null  int64   
 4   genero                       11140 non-null  object  
 5   cor_raca                     11140 non-null  object  
 6   grau_instrucao               11140 non-null  object  
 7   faixa_etaria                 3002 non-null   category
 8   caracterizacao_orgao_gestor  11140 non-null  object  
dtypes: category(1), int64(2), object(6)
memory usage: 794.4+ KB


In [ ]:
# Define the BigQuery table schema with Portuguese descriptions
schema=[bigquery.SchemaField('ano','INTEGER',description='Ano de referência da observação'),
        bigquery.SchemaField('sigla_uf','STRING',description='Sigla da Unidade da Federação referente municipio'),
        bigquery.SchemaField('nome_municipio','STRING',description='Nome do município'),
        bigquery.SchemaField('id_municipio','INTEGER',description='Código do IBGE do município'),
        bigquery.SchemaField('caracterizacao_orgao_gestor','STRING',description='Caracterização do órgão no qual o gestor está'),
        bigquery.SchemaField('genero','STRING',description='Gênero autodeclarado ou não'),
        bigquery.SchemaField('cor_raca','STRING',description='Raça/cor da pessoa observada'),
        bigquery.SchemaField('grau_instrucao','STRING',description='Escolaridade da pessoa ou do vínculo observado com detalhamento na pós-graduação'),
        bigquery.SchemaField('faixa_etaria','STRING',description='faixa etária da observação')
        ]
# Initialize BigQuery client connection
client = bigquery.Client(project='repositoriodedadosgpsp')

# Create reference to target dataset
dataset_ref = client.dataset('cargos_lideranca')

# Create reference to target table with standardized naming convention:
# FONTE_algo_intuitivo_dado (MUNIC_quantidade_vinculos_mapa_v1)
table_ref = dataset_ref.table('MUNIC_perfil_gestor_seguranca_publica_tipo_orgao_v6')

# Configure the load job with our schema definition
job_config = bigquery.LoadJobConfig(
    schema=schema,
    # Optional parameters (commented out):
    # write_disposition="WRITE_TRUNCATE",  # Overwrites table if exists
    # create_disposition="CREATE_IF_NEEDED"  # Default behavior
)

# Execute the load job to upload DataFrame to BigQuery
job = client.load_table_from_dataframe(
    dataframe=df_final,
    destination=table_ref,
    job_config=job_config
)

# Wait for the job to complete
job.result()

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


LoadJob<project=repositoriodedadosgpsp, location=US, id=cf0f5982-07c8-4ceb-8a16-da3b7ce9a1d1>